In [0]:
# Libraries this notebook needs:
#   azure-identity              -> logs in as the service principal
#   azure-storage-file-datalake -> manages ADLS containers and folders
%pip install azure-identity azure-storage-file-datalake
dbutils.library.restartPython()   # restart Python so the new libraries load

In [0]:
# Non-secret settings. Later these move into src/config.py.
STORAGE_ACCOUNT = "stuberetlabhi01"
SECRET_SCOPE    = "uber-etl-scope"
CONTAINERS      = ["raw", "bronze", "silver", "gold", "quarantine", "reports"]

# Databricks already provides a Spark session named `spark`.
# getOrCreate() just returns that existing session.
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
print("Spark version:", spark.version)

In [0]:
# Read the service principal's credentials from the secret scope.
# They are never written in code. If you print one, Databricks shows [REDACTED].
client_id     = dbutils.secrets.get(scope=SECRET_SCOPE, key="sp-client-id")
tenant_id     = dbutils.secrets.get(scope=SECRET_SCOPE, key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope=SECRET_SCOPE, key="sp-client-secret")

# Tell Spark to log in to our storage account with OAuth as the service principal.
host = f"{STORAGE_ACCOUNT}.dfs.core.windows.net"
spark.conf.set(f"fs.azure.account.auth.type.{host}", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{host}",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{host}", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{host}", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{host}",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

def abfss(container, path=""):
    """Build an ADLS path, e.g. abfss://raw@stuberetlabhi01.dfs.core.windows.net/uber_rides"""
    return f"abfss://{container}@{host}/{path}"

print("ADLS access configured for", host)

In [0]:
# "Idempotent" means running this twice is safe: it only creates what's missing.
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

credential = ClientSecretCredential(tenant_id, client_id, client_secret)
service = DataLakeServiceClient(f"https://{host}", credential=credential)

existing = {fs.name for fs in service.list_file_systems()}
for c in CONTAINERS:
    if c in existing:
        print(f"✓ {c} already exists")
    else:
        service.create_file_system(c)
        print(f"+ created {c}")

In [0]:
# Folder layout from the PDF (section 3).
folders = {
    "raw":        ["uber_rides"],
    "bronze":     ["uber_rides"],
    "silver":     ["uber_rides"],
    "gold":       [""],
    "quarantine": ["uber_rides"],
    "reports":    [""],
}
for container, paths in folders.items():
    for p in paths:
        dbutils.fs.mkdirs(abfss(container, p))
    # Listing proves Spark can actually reach the container.
    print(f"{container}: {[f.name for f in dbutils.fs.ls(abfss(container))]}")

In [0]:
# 1) Read the raw CSV, just to count rows.
raw_df = spark.read.option("header", True).csv(abfss("raw", "uber_rides/"))
print("Raw rows:", raw_df.count(), "| columns:", len(raw_df.columns))

# 2) Write a tiny Delta table, read it back, then delete it.
test_path = abfss("reports", "_setup_test")
spark.range(5).write.format("delta").mode("overwrite").save(test_path)
print("Delta round-trip rows:", spark.read.format("delta").load(test_path).count())
dbutils.fs.rm(test_path, recurse=True)
print("✓ Read/write to ADLS works")

In [0]:
# Unity Catalog workspace: create the schema in the session's default catalog.
# Our Delta data stays in ADLS (path-based); this schema is the project's namespace.
current_catalog = spark.sql("SELECT current_catalog()").first()[0]
print("Default catalog:", current_catalog)

spark.sql("CREATE SCHEMA IF NOT EXISTS uber_etl COMMENT 'Uber taxi agentic ETL project'")
spark.sql("SHOW SCHEMAS").filter("databaseName = 'uber_etl'").show()
print(f"✓ Schema ready: {current_catalog}.uber_etl")

In [ ]:
# Test the Foundry GPT endpoint (Phase 2). Needs secrets foundry-endpoint and foundry-deployment
# in Key Vault, and the role "Cognitive Services OpenAI User" for sp-uber-etl on the Foundry resource.
from src.agents import FoundryClient
llm = FoundryClient(spark)
print(llm.ask_json("You are a test assistant.", {"task": 'Reply with {"status": "ok"}'}))
print("Tokens used:", llm.tokens_used)